In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
import joblib

Loading Dataset

In [33]:
data = pd.read_csv('irrigation_dataset.csv')

data = data.drop(columns=['Water_Usage(cubic_meters)'])

          Crop  Season Soil_Type  Area(ha)  Water_Usage(cubic_meters)  \
0         rice    rabi    clayey      3.86                    5291.18   
1       tomato  kharif    clayey      1.02                     714.12   
2    groundnut    rabi     black      0.81                     577.98   
3    sugarcane  kharif     black      4.19                    7143.69   
4       cotton  summer     sandy      1.30                    1104.56   
..         ...     ...       ...       ...                        ...   
995      onion  summer     loamy      3.17                    1748.86   
996       rice    rabi    clayey      1.39                    1739.09   
997      maize  summer     loamy      1.95                    1258.58   
998      wheat  summer     black      2.16                    1800.35   
999     tomato  summer     black      3.41                    1960.51   

    Irrigation_Type  
0           surface  
1              drip  
2           surface  
3           surface  
4            

Defining x and y features

In [34]:

categorical_features = ['Crop', 'Season', 'Soil_Type']
data = pd.get_dummies(data, columns=categorical_features)

label_encoder = LabelEncoder()
data['Irrigation_Type'] = label_encoder.fit_transform(data['Irrigation_Type'])


In [35]:
X = data.drop(columns=['Irrigation_Type'])
y = data['Irrigation_Type']

Encoding Y

In [36]:

y_encoded = to_categorical(y)

Splittibng x and y into training and testing

In [37]:


X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y
)


Creating Model

In [61]:
model = Sequential()
model.add(Dense(64, input_dim=X.shape[1], activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(y_encoded.shape[1], activation='softmax')) 

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


model.fit(X_train, y_train, epochs=100, batch_size=16, verbose=1)

loss, accuracy = model.evaluate(X_test, y_test)
print(f"\n✅ Test Accuracy: {accuracy:.4f}")

RandomForestClassifier(random_state=42)

Doing Predictions

In [62]:

y_pred_probs = model.predict(X_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

y_pred_labels = label_encoder.inverse_transform(y_pred_classes)
y_true_labels = label_encoder.inverse_transform(y_true_classes)

print("\n📊 Classification Report:")
print(classification_report(y_true_labels, y_pred_labels))

Saving Model

In [65]:
model.save("irrigation_type_ann_model.keras")
joblib.dump(label_encoder, "irrigation_type_label_encoder.pkl")
joblib.dump(X.columns, "irrigation_type_columns.pkl")



Model and Label Encoder saved successfully!
